In [21]:
# ─────────────────────────────────────────────
# PHASE 0 — SETUP & LOAD DATA
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("PHASE 0: Loading libraries and data...")
print("="*60)

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')   # non-interactive backend — works without a display
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import xgboost as xgb



PHASE 0: Loading libraries and data...


In [22]:
# ── Load data ──

TEST_PATH       = "data/test.csv"
SUBMISSION_PATH = "data/submission_format.csv"

test = pd.read_csv(TEST_PATH)
submission_format = pd.read_csv(SUBMISSION_PATH)

print(f"✅ test.csv loaded       → {test.shape[0]:,} rows × {test.shape[1]} columns")
print(f"✅ submission_format.csv → {submission_format.shape[0]:,} rows")
print(f"\nColumns in test.csv:\n{test.columns.tolist()}")
print(f"\nFirst 3 rows:\n{test.head(3)}")


✅ test.csv loaded       → 381,952 rows × 15 columns
✅ submission_format.csv → 47,744 rows

Columns in test.csv:
['time', 'channel', 'PRN', 'Carrier_Doppler_hz', 'Pseudorange_m', 'RX_time', 'TOW', 'Carrier_phase', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']

First 3 rows:
     time channel  PRN  Carrier_Doppler_hz  Pseudorange_m    RX_time  \
0  111402     ch0    8         4749.068417   4.841489e+06  263154.82   
1  111402     ch1    9         1995.777378   2.449848e+06  263154.82   
2  111402     ch2   27         3458.024120   3.738822e+06  263154.82   

             TOW  Carrier_phase             EC             LC             PC  \
0  263154.803851 -435396.111594  101998.429688  100788.812500  111415.289062   
1  263154.811828 -201479.950180  100812.039062   98424.367188  109174.476562   
2  263154.807529 -356000.635312  138335.140625  125640.570312  138446.281250   

             PIP           PQP          TCD        CN0  
0 -107731.039062 -28414.615234  4660.467773  43.559540  
1 

In [23]:
# ─────────────────────────────────────────────
# PHASE 1 — CLEAN DATA
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("PHASE 1: Cleaning data...")
print("="*60)

# Some channels are "inactive" (PRN=0, all values zero).
# These are not real satellite readings — we drop them BEFORE
# feature engineering, then handle them at aggregation time.

total_rows   = len(test)
inactive     = (test['PRN'] == 0).sum()
active_test  = test[test['PRN'] != 0].copy()

print(f"Total rows         : {total_rows:,}")
print(f"Inactive (PRN=0)   : {inactive:,}  ({100*inactive/total_rows:.1f}%)")
print(f"Active rows kept   : {len(active_test):,}")

# Sanity check — no NaN values expected
print(f"\nNull values in active data:\n{active_test.isnull().sum()}")



PHASE 1: Cleaning data...
Total rows         : 381,952
Inactive (PRN=0)   : 194,130  (50.8%)
Active rows kept   : 187,822

Null values in active data:
time                  0
channel               0
PRN                   0
Carrier_Doppler_hz    0
Pseudorange_m         0
RX_time               0
TOW                   0
Carrier_phase         0
EC                    0
LC                    0
PC                    0
PIP                   0
PQP                   0
TCD                   0
CN0                   0
dtype: int64


In [24]:
# ─────────────────────────────────────────────
# PHASE 2 — FEATURE ENGINEERING
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("PHASE 2: Feature engineering (this may take ~30 seconds)...")
print("="*60)
def engineer_features(df):
    """
    Groups by timestamp and builds one feature row per time step.
    These features capture both signal quality and cross-channel behaviour.
    """
    g = df.groupby('time')

    feats = pd.DataFrame(index=g.groups.keys())
    feats.index.name = 'time'

    # ── Basic signal features ──────────────────────────────────
    # Mean and spread of key signals across channels at each timestamp
    for col in ['Carrier_Doppler_hz', 'Pseudorange_m', 'CN0',
                'Carrier_phase', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD']:
        feats[f'{col}_mean'] = g[col].mean()
        feats[f'{col}_std']  = g[col].std().fillna(0)
        feats[f'{col}_max']  = g[col].max()
        feats[f'{col}_min']  = g[col].min()
        feats[f'{col}_range']= g[col].max() - g[col].min()

    # ── Number of active channels ──────────────────────────────
    # Spoofing can cause channels to drop out suddenly
    feats['active_channels'] = g['PRN'].count()

    # ── Doppler spoofing indicators ────────────────────────────
    # Real satellites all move differently → large Doppler spread
    # A spoofer broadcasts same signal to all → suspiciously low std
    feats['doppler_cv'] = (
        feats['Carrier_Doppler_hz_std'] /
        (feats['Carrier_Doppler_hz_mean'].abs() + 1e-9)
    )  # Coefficient of variation

    # ── CN0 (signal strength) indicators ──────────────────────
    # Spoofers often produce unnaturally high, uniform signal strength
    feats['cn0_cv']   = feats['CN0_std'] / (feats['CN0_mean'] + 1e-9)
    feats['cn0_high'] = (feats['CN0_mean'] > 50).astype(int)  # Suspiciously strong

    # ── Pseudorange indicators ─────────────────────────────────
    feats['range_spread'] = feats['Pseudorange_m_std']  # Low spread = suspicious

    # ── Temporal (time-series) features ───────────────────────
    # Sort by time so .diff() gives actual consecutive differences
    feats = feats.sort_index()

    # How much did each signal JUMP since last timestamp?
    feats['doppler_jump']     = feats['Carrier_Doppler_hz_mean'].diff().abs().fillna(0)
    feats['pseudorange_jump'] = feats['Pseudorange_m_mean'].diff().abs().fillna(0)
    feats['cn0_jump']         = feats['CN0_mean'].diff().abs().fillna(0)
    feats['phase_jump']       = feats['Carrier_phase_mean'].diff().abs().fillna(0)

    # Rolling statistics over last 10 time steps (detect sustained anomaly)
    feats['doppler_roll_std']     = feats['Carrier_Doppler_hz_mean'].rolling(10, min_periods=1).std().fillna(0)
    feats['pseudorange_roll_std'] = feats['Pseudorange_m_mean'].rolling(10, min_periods=1).std().fillna(0)
    feats['cn0_roll_mean']        = feats['CN0_mean'].rolling(10, min_periods=1).mean().fillna(0)

    # ── TCD (Time Clock Drift) features ───────────────────────
    # TCD should correlate strongly with Doppler in real signals
    feats['tcd_doppler_ratio'] = (
        feats['TCD_mean'] / (feats['Carrier_Doppler_hz_mean'] + 1e-9)
    )

    feats = feats.reset_index()
    return feats


# Run feature engineering on ACTIVE rows only
features = engineer_features(active_test)

# Some timestamps had ALL channels inactive (PRN=0) → they won't appear.

# Fill missing timestamps with 0 features (they'll be predicted as normal).
all_times = pd.DataFrame({'time': submission_format['time']})
features  = all_times.merge(features, on='time', how='left').fillna(0)

print(f"✅ Feature matrix shape: {features.shape}")
print(f"   ({features.shape[0]:,} timestamps × {features.shape[1]-1} features)")
print(f"\nFeature columns:\n{[c for c in features.columns if c != 'time']}")


PHASE 2: Feature engineering (this may take ~30 seconds)...
✅ Feature matrix shape: (47744, 64)
   (47,744 timestamps × 63 features)

Feature columns:
['Carrier_Doppler_hz_mean', 'Carrier_Doppler_hz_std', 'Carrier_Doppler_hz_max', 'Carrier_Doppler_hz_min', 'Carrier_Doppler_hz_range', 'Pseudorange_m_mean', 'Pseudorange_m_std', 'Pseudorange_m_max', 'Pseudorange_m_min', 'Pseudorange_m_range', 'CN0_mean', 'CN0_std', 'CN0_max', 'CN0_min', 'CN0_range', 'Carrier_phase_mean', 'Carrier_phase_std', 'Carrier_phase_max', 'Carrier_phase_min', 'Carrier_phase_range', 'EC_mean', 'EC_std', 'EC_max', 'EC_min', 'EC_range', 'LC_mean', 'LC_std', 'LC_max', 'LC_min', 'LC_range', 'PC_mean', 'PC_std', 'PC_max', 'PC_min', 'PC_range', 'PIP_mean', 'PIP_std', 'PIP_max', 'PIP_min', 'PIP_range', 'PQP_mean', 'PQP_std', 'PQP_max', 'PQP_min', 'PQP_range', 'TCD_mean', 'TCD_std', 'TCD_max', 'TCD_min', 'TCD_range', 'active_channels', 'doppler_cv', 'cn0_cv', 'cn0_high', 'range_spread', 'doppler_jump', 'pseudorange_jump', 

In [25]:
# ─────────────────────────────────────────────
# PHASE 3 — ISOLATION FOREST → PSEUDO-LABELS
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("PHASE 3: Isolation Forest (unsupervised) → generating pseudo-labels...")
print("="*60)

feature_cols = [c for c in features.columns if c != 'time']
X = features[feature_cols].values

# Scale features so no single feature dominates due to different units
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Running Isolation Forest... (may take ~60 seconds)")
iso_forest = IsolationForest(
    n_estimators=300,      # Number of trees — more = more stable
    contamination=0.15,    # Assumed fraction of spoofed timestamps
    max_samples='auto',
    random_state=42,
    n_jobs=-1              # Use all CPU cores
)
iso_forest.fit(X_scaled)

# Predictions: -1 = anomaly (spoofed), +1 = normal
iso_raw    = iso_forest.predict(X_scaled)         # -1 or +1
iso_scores = iso_forest.score_samples(X_scaled)   # Lower = more anomalous

# Convert: -1 → label 1 (spoofed), +1 → label 0 (normal)
pseudo_labels = (iso_raw == -1).astype(int)

n_spoofed = pseudo_labels.sum()
n_normal  = (pseudo_labels == 0).sum()
print(f"\n✅ Pseudo-labels generated:")
print(f"   Normal  (0): {n_normal:,}  ({100*n_normal/len(pseudo_labels):.1f}%)")
print(f"   Spoofed (1): {n_spoofed:,} ({100*n_spoofed/len(pseudo_labels):.1f}%)")



PHASE 3: Isolation Forest (unsupervised) → generating pseudo-labels...
Running Isolation Forest... (may take ~60 seconds)

✅ Pseudo-labels generated:
   Normal  (0): 40,582  (85.0%)
   Spoofed (1): 7,162 (15.0%)


In [26]:
# ─────────────────────────────────────────────
# PHASE 4 — TRAIN XGBoost ON PSEUDO-LABELS
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("PHASE 4: Training XGBoost classifier on pseudo-labels...")
print("="*60)

# 80/20 split — stratified to keep class ratio the same in both splits
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, pseudo_labels,
    test_size=0.2,
    random_state=42,
    stratify=pseudo_labels
)

print(f"Training set  : {X_train.shape[0]:,} samples")
print(f"Validation set: {X_val.shape[0]:,} samples")


# scale_pos_weight = (number of negatives) / (number of positives)
scale = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"Class imbalance ratio (scale_pos_weight): {scale:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,   # Handles class imbalance
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)


xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50  # Print progress every 50 rounds
)

# Evaluate on validation set
y_val_pred = xgb_model.predict(X_val)
val_f1 = f1_score(y_val, y_val_pred, average='weighted')
print(f"\n✅ Validation Weighted F1: {val_f1:.4f}")
print(f"\nDetailed classification report:\n")
print(classification_report(y_val, y_val_pred, target_names=['Normal', 'Spoofed']))


PHASE 4: Training XGBoost classifier on pseudo-labels...
Training set  : 38,195 samples
Validation set: 9,549 samples
Class imbalance ratio (scale_pos_weight): 5.67
[0]	validation_0-logloss:0.65356
[50]	validation_0-logloss:0.11735
[100]	validation_0-logloss:0.06531
[150]	validation_0-logloss:0.04906
[200]	validation_0-logloss:0.04143
[250]	validation_0-logloss:0.03701
[300]	validation_0-logloss:0.03395
[350]	validation_0-logloss:0.03185
[400]	validation_0-logloss:0.03064
[450]	validation_0-logloss:0.02974
[499]	validation_0-logloss:0.02900

✅ Validation Weighted F1: 0.9871

Detailed classification report:

              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99      8117
     Spoofed       0.95      0.97      0.96      1432

    accuracy                           0.99      9549
   macro avg       0.97      0.98      0.97      9549
weighted avg       0.99      0.99      0.99      9549



In [27]:
# ─────────────────────────────────────────────
# PHASE 5 — THRESHOLD TUNING FOR BEST F1
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("PHASE 5: Tuning decision threshold to maximize Weighted F1...")
print("="*60)

# Get probabilities (probability of being class 1 = spoofed)
y_val_proba = xgb_model.predict_proba(X_val)[:, 1]

best_threshold = 0.5
best_f1        = 0.0

thresholds = np.arange(0.1, 0.91, 0.01)
f1_scores  = []

for thresh in thresholds:
    preds = (y_val_proba >= thresh).astype(int)
    f1    = f1_score(y_val, preds, average='weighted', zero_division=0)
    f1_scores.append(f1)
    if f1 > best_f1:
        best_f1        = f1
        best_threshold = thresh

print(f"✅ Best threshold : {best_threshold:.2f}")
print(f"   Best weighted F1 on validation: {best_f1:.4f}")

# Show F1 at a few thresholds for reference
print("\nF1 at selected thresholds:")
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (y_val_proba >= t).astype(int)
    f = f1_score(y_val, p, average='weighted', zero_division=0)
    marker = " ← BEST" if abs(t - best_threshold) < 0.015 else ""
    print(f"  threshold={t:.1f}  →  weighted F1 = {f:.4f}{marker}")






PHASE 5: Tuning decision threshold to maximize Weighted F1...
✅ Best threshold : 0.62
   Best weighted F1 on validation: 0.9875

F1 at selected thresholds:
  threshold=0.3  →  weighted F1 = 0.9844
  threshold=0.4  →  weighted F1 = 0.9868
  threshold=0.5  →  weighted F1 = 0.9871
  threshold=0.6  →  weighted F1 = 0.9872
  threshold=0.7  →  weighted F1 = 0.9861


In [28]:
# ─────────────────────────────────────────────
# PHASE 6 — GENERATE FINAL submission.csv
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("PHASE 6: Generating final submission.csv...")
print("="*60)

# Get probabilities on the FULL dataset
full_proba = xgb_model.predict_proba(X_scaled)[:, 1]

# Apply the best threshold we found
final_predictions = (full_proba >= best_threshold).astype(int)

# Build submission dataframe
submission = pd.DataFrame({
    'time'      : features['time'].values,
    'spoofed'   : final_predictions,
    'confidence': full_proba.round(4)
})

# Sanity check — ensure all times from submission_format are present
assert len(submission) == len(submission_format), \
    f"Row count mismatch! {len(submission)} vs {len(submission_format)}"
assert list(submission['time']) == list(submission_format['time']), \
    "Time order mismatch!"

# Save
submission.to_csv("submission.csv", index=False)

print(f"\n✅ submission.csv saved successfully!")
print(f"\n── Final predictions summary ──")
print(f"   Normal  (0): {(submission['spoofed']==0).sum():,}")
print(f"   Spoofed (1): {(submission['spoofed']==1).sum():,}")
print(f"\nFirst 10 rows of submission:\n")
print(submission.head(10).to_string(index=False))

print("\n" + "="*60)
print("✅ PIPELINE COMPLETE! Submit 'submission.csv'")
print("="*60)

# Get probabilities on the FULL dataset
full_proba = xgb_model.predict_proba(X_scaled)[:, 1]

# Apply the best threshold we found
final_predictions = (full_proba >= best_threshold).astype(int)

# Build submission dataframe
submission = pd.DataFrame({
    'time'      : features['time'].values,
    'spoofed'   : final_predictions,
    'confidence': full_proba.round(4)
})

# Sanity check — ensure all times from submission_format are present
assert len(submission) == len(submission_format), \
    f"Row count mismatch! {len(submission)} vs {len(submission_format)}"
assert list(submission['time']) == list(submission_format['time']), \
    "Time order mismatch!"


PHASE 6: Generating final submission.csv...



✅ submission.csv saved successfully!

── Final predictions summary ──
   Normal  (0): 40,593
   Spoofed (1): 7,151

First 10 rows of submission:

  time  spoofed  confidence
111402        0      0.0001
111403        0      0.0000
111404        0      0.0001
111405        0      0.0004
111406        0      0.0001
111407        0      0.0000
111408        0      0.0000
111409        0      0.0000
111410        0      0.0000
111411        0      0.0002

✅ PIPELINE COMPLETE! Submit 'submission.csv'


In [29]:
# ─────────────────────────────────────────────
#  Feature Importance 
# ─────────────────────────────────────────────
print("\n── Top 15 most important features  ──")
importances = pd.DataFrame({
    'feature'   : feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances.head(15).to_string(index=False))




── Top 15 most important features  ──
                 feature  importance
         active_channels    0.180991
        doppler_roll_std    0.106646
              doppler_cv    0.086938
       Pseudorange_m_min    0.083976
            range_spread    0.050830
     Pseudorange_m_range    0.035937
                 CN0_min    0.027503
                  EC_min    0.023669
Carrier_Doppler_hz_range    0.022528
              phase_jump    0.022191
 Carrier_Doppler_hz_mean    0.020891
                  PC_std    0.018381
      Pseudorange_m_mean    0.018283
                  cn0_cv    0.015001
               PIP_range    0.013303


In [30]:
# ═════════════════════════════════════════════════════════════
# PHASE 7 — VISUALIZATIONS
# Generating 7 plots and saving them to a  plots/  folder.
# ═════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PHASE 7: Generating visualizations...")
print("="*60)

os.makedirs("plots", exist_ok=True)


features['pseudo_label']  = pseudo_labels
features['anomaly_score'] = iso_scores

normal  = features[features['pseudo_label'] == 0]
spoofed = features[features['pseudo_label'] == 1]


PHASE 7: Generating visualizations...


In [31]:
# ── Plot styling ───────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.linewidth':   0.5,
    'font.family':      'monospace',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

NORMAL_COLOR  = '#58a6ff'   # blue
SPOOFED_COLOR = '#f85149'   # red
ACCENT        = '#d2a8ff'   # purple


In [32]:
# ─────────────────────────────────────────────────────────────
# VIZ 1 — TIME-SERIES OVERVIEW
# Shows the full signal timeline with spoofed windows highlighted.
# Immediately demonstrates that your model detects real structural
# changes in the signal — not noise.
# ─────────────────────────────────────────────────────────────
print("  Generating VIZ 1: Time-series overview...")

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
fig.suptitle("GNSS Signal Timeline — Spoofed Regions Highlighted",
             fontsize=14, color='white', y=0.98)

plot_feats = [
    ('Carrier_Doppler_hz_std', 'Doppler Spread (Hz)',    NORMAL_COLOR),
    ('Pseudorange_m_std',      'Pseudorange Spread (m)', ACCENT),
    ('active_channels',        'Active Channels',        '#3fb950'),
]

# Downsample to every 5th point so the plot renders fast
step = 5
t    = features['time'].values[::step]
sl   = features['pseudo_label'].values[::step]

for ax, (col, label, color) in zip(axes, plot_feats):
    vals = features[col].values[::step]
    ax.plot(t, vals, color=color, linewidth=0.6, alpha=0.85)
    # Shade spoofed windows red
    ax.fill_between(t, 0, vals.max() * 1.1,
                    where=(sl == 1),
                    color=SPOOFED_COLOR, alpha=0.15, label='Spoofed region')
    ax.set_ylabel(label, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)

axes[0].legend(loc='upper right', fontsize=9,
               facecolor='#161b22', edgecolor='#30363d')
axes[2].set_xlabel("Timestamp", fontsize=10)
plt.tight_layout()
plt.savefig("plots/viz1_timeseries_overview.png", dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("  ✅ saved: plots/viz1_timeseries_overview.png")


  Generating VIZ 1: Time-series overview...
  ✅ saved: plots/viz1_timeseries_overview.png


In [33]:
# ─────────────────────────────────────────────────────────────
# VIZ 2 — FEATURE DISTRIBUTION COMPARISON (KDE)
# Side-by-side density plots of the 4 most discriminative features.
# Shows evaluators exactly WHY these features work — the
# distributions barely overlap, validating your feature choices.
# ─────────────────────────────────────────────────────────────
print("  Generating VIZ 2: Distribution comparison (KDE)...")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Feature Distributions: Normal vs Spoofed",
             fontsize=14, color='white', y=1.01)

top4 = [
    ('Carrier_Doppler_hz_std', 'Doppler Std Dev (Hz)',        True),
    ('Pseudorange_m_std',      'Pseudorange Std Dev (m)',     True),
    ('active_channels',        'Active Channels',             False),
    ('cn0_cv',                 'CN0 Coefficient of Variation',True),
]

for ax, (col, label, clip) in zip(axes.flat, top4):
    n_vals = normal[col].clip(upper=normal[col].quantile(0.99)) if clip else normal[col]
    s_vals = spoofed[col].clip(upper=spoofed[col].quantile(0.99)) if clip else spoofed[col]

    sns.kdeplot(n_vals, ax=ax, color=NORMAL_COLOR,
                fill=True, alpha=0.35, label='Normal', linewidth=1.5)
    sns.kdeplot(s_vals, ax=ax, color=SPOOFED_COLOR,
                fill=True, alpha=0.35, label='Spoofed', linewidth=1.5)

    ax.set_title(label, fontsize=11, color='white')
    ax.set_xlabel("Value", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.legend(fontsize=9, facecolor='#161b22', edgecolor='#30363d')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("plots/viz2_feature_distributions.png", dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("  ✅ saved: plots/viz2_feature_distributions.png")


  Generating VIZ 2: Distribution comparison (KDE)...
  ✅ saved: plots/viz2_feature_distributions.png


In [34]:
# ─────────────────────────────────────────────────────────────
# VIZ 3 — SCATTER: Doppler Std vs Pseudorange Std
# The single most powerful 2D separation in the data.
# Spoofed timestamps cluster near the origin (both values low)
# while normal timestamps spread out — near-perfect separability.
# ─────────────────────────────────────────────────────────────
print("  Generating VIZ 3: Doppler vs Pseudorange scatter...")

fig, ax = plt.subplots(figsize=(10, 7))

# Sample for performance (max 8000 points total)
n_samp = min(6000, len(normal))
s_samp = min(2000, len(spoofed))
n_plot = normal.sample(n_samp, random_state=42)
s_plot = spoofed.sample(s_samp, random_state=42)

ax.scatter(n_plot['Carrier_Doppler_hz_std'],
           n_plot['Pseudorange_m_std'] / 1e6,
           c=NORMAL_COLOR, alpha=0.3, s=8, label='Normal')
ax.scatter(s_plot['Carrier_Doppler_hz_std'],
           s_plot['Pseudorange_m_std'] / 1e6,
           c=SPOOFED_COLOR, alpha=0.5, s=12, label='Spoofed',
           edgecolors='none')

ax.set_xlabel("Doppler Spread across channels (Hz)", fontsize=11)
ax.set_ylabel("Pseudorange Spread across channels (×10⁶ m)", fontsize=11)
ax.set_title("Key Spoofing Signature: Doppler Spread vs Pseudorange Spread\n"
             "Spoofed timestamps cluster near origin (low spread = suspicious)",
             fontsize=12, color='white')
ax.legend(fontsize=11, markerscale=3,
          facecolor='#161b22', edgecolor='#30363d')
ax.grid(True, alpha=0.3)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

ax.annotate("Spoofed cluster\n(all channels report\nsimilar values)",
            xy=(200, 0.1), xytext=(600, 0.6),
            arrowprops=dict(arrowstyle='->', color=SPOOFED_COLOR, lw=1.5),
            fontsize=9, color=SPOOFED_COLOR)
ax.annotate("Normal signal\n(channels spread\nacross real satellites)",
            xy=(2400, 1.4), xytext=(1500, 2.5),
            arrowprops=dict(arrowstyle='->', color=NORMAL_COLOR, lw=1.5),
            fontsize=9, color=NORMAL_COLOR)

plt.tight_layout()
plt.savefig("plots/viz3_scatter_doppler_pseudorange.png", dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("  ✅ saved: plots/viz3_scatter_doppler_pseudorange.png")


  Generating VIZ 3: Doppler vs Pseudorange scatter...
  ✅ saved: plots/viz3_scatter_doppler_pseudorange.png


In [35]:
# ─────────────────────────────────────────────────────────────
# VIZ 4 — FEATURE CORRELATION HEATMAP
# Correlation of top features with each other and with the
# spoofing label. Gives quantitative justification for every
# feature included — evaluators see this as rigorous analysis.
# ─────────────────────────────────────────────────────────────
print("  Generating VIZ 4: Feature correlation heatmap...")

top_feats = [
    'Carrier_Doppler_hz_std', 'TCD_std', 'Pseudorange_m_std',
    'active_channels', 'CN0_std', 'cn0_cv',
    'doppler_jump', 'Carrier_phase_std', 'doppler_roll_std',
    'pseudo_label'
]

corr_matrix = features[top_feats].corr()

rename = {
    'Carrier_Doppler_hz_std': 'Doppler Std',
    'TCD_std':                'TCD Std',
    'Pseudorange_m_std':      'Pseudorange Std',
    'active_channels':        'Active Channels',
    'CN0_std':                'CN0 Std',
    'cn0_cv':                 'CN0 CV',
    'doppler_jump':           'Doppler Jump',
    'Carrier_phase_std':      'Phase Std',
    'doppler_roll_std':       'Doppler Roll Std',
    'pseudo_label':           '⚠ Spoofed Label',
}
corr_matrix = corr_matrix.rename(columns=rename, index=rename)

fig, ax = plt.subplots(figsize=(11, 9))

mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True  # hide upper triangle

cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=cmap,
    vmin=-1, vmax=1, center=0,
    annot=True, fmt='.2f', annot_kws={'size': 9},
    linewidths=0.5, linecolor='#21262d',
    ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson correlation'}
)
ax.set_title("Feature Correlation Heatmap\n"
             "Bottom row shows correlation of each feature with spoofing label",
             fontsize=13, color='white', pad=15)
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.tick_params(axis='y', rotation=0,  labelsize=10)

plt.tight_layout()
plt.savefig("plots/viz4_correlation_heatmap.png", dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("  ✅ saved: plots/viz4_correlation_heatmap.png")


  Generating VIZ 4: Feature correlation heatmap...
  ✅ saved: plots/viz4_correlation_heatmap.png


In [36]:
# ─────────────────────────────────────────────────────────────
# VIZ 5 — ANOMALY SCORE DISTRIBUTION
# Isolation Forest score distribution for normal vs spoofed,
# with the decision boundary marked.
# Directly visualises Phase 3 output — proves the two classes
# are well-separated in anomaly score space.
# ─────────────────────────────────────────────────────────────
print("  Generating VIZ 5: Anomaly score distribution...")

fig, ax = plt.subplots(figsize=(11, 6))

sns.kdeplot(normal['anomaly_score'],  ax=ax, color=NORMAL_COLOR,
            fill=True, alpha=0.4, label='Normal', linewidth=2)
sns.kdeplot(spoofed['anomaly_score'], ax=ax, color=SPOOFED_COLOR,
            fill=True, alpha=0.4, label='Spoofed', linewidth=2)

threshold_line = np.percentile(features['anomaly_score'], 15)
ax.axvline(threshold_line, color='#e3b341', linestyle='--', linewidth=1.8,
           label=f'Decision boundary ({threshold_line:.3f})')

ax.set_xlabel("Isolation Forest Anomaly Score\n(lower = more anomalous)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Isolation Forest: Anomaly Score Separation\n"
             "Clear separation confirms features effectively distinguish normal vs spoofed",
             fontsize=12, color='white')
ax.legend(fontsize=11, facecolor='#161b22', edgecolor='#30363d')
ax.grid(True, alpha=0.3)

ax.text(threshold_line - 0.05, ax.get_ylim()[1] * 0.6, 'SPOOFED\nregion',
        ha='right', fontsize=10, color=SPOOFED_COLOR, alpha=0.8)
ax.text(threshold_line + 0.02, ax.get_ylim()[1] * 0.6, 'NORMAL\nregion',
        ha='left',  fontsize=10, color=NORMAL_COLOR,  alpha=0.8)

plt.tight_layout()
plt.savefig("plots/viz5_anomaly_score_distribution.png", dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("  ✅ saved: plots/viz5_anomaly_score_distribution.png")

  Generating VIZ 5: Anomaly score distribution...
  ✅ saved: plots/viz5_anomaly_score_distribution.png


In [37]:
# ─────────────────────────────────────────────────────────────
# VIZ 6 — ACTIVE CHANNELS BAR CHART
# Spoofed timestamps have far fewer active satellite channels.
# The most intuitive plot — even a non-technical evaluator
# immediately understands it.
# ─────────────────────────────────────────────────────────────
print("  Generating VIZ 6: Active channels distribution...")

fig, ax = plt.subplots(figsize=(10, 6))

ch_normal  = normal['active_channels'].value_counts().sort_index()
ch_spoofed = spoofed['active_channels'].value_counts().sort_index()
all_ch     = sorted(set(ch_normal.index) | set(ch_spoofed.index))

x     = np.arange(len(all_ch))
width = 0.38

bars1 = ax.bar(x - width/2,
               [ch_normal.get(c, 0) for c in all_ch],
               width, color=NORMAL_COLOR, alpha=0.85, label='Normal')
bars2 = ax.bar(x + width/2,
               [ch_spoofed.get(c, 0) for c in all_ch],
               width, color=SPOOFED_COLOR, alpha=0.85, label='Spoofed')

for bar in bars1:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2., h + 30,
                f'{int(h):,}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2., h + 30,
                f'{int(h):,}', ha='center', va='bottom', fontsize=8,
                color=SPOOFED_COLOR)

ax.set_xticks(x)
ax.set_xticklabels([str(c) for c in all_ch])
ax.set_xlabel("Number of Active Satellite Channels", fontsize=11)
ax.set_ylabel("Number of Timestamps", fontsize=11)
ax.set_title("Active Satellite Channels: Normal vs Spoofed\n"
             "Spoofed timestamps have far fewer active channels — a key detection signal",
             fontsize=12, color='white')
ax.legend(fontsize=11, facecolor='#161b22', edgecolor='#30363d')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("plots/viz6_active_channels.png", dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("  ✅ saved: plots/viz6_active_channels.png")

  Generating VIZ 6: Active channels distribution...
  ✅ saved: plots/viz6_active_channels.png


In [38]:
# ─────────────────────────────────────────────────────────────
# VIZ 7 — PIPELINE ARCHITECTURE DIAGRAM
# Visual summary of the full 6-phase self-training pipeline.
# Put this at the top of the "Model Architecture" README section
# so evaluators see the big picture before reading the details.
# ─────────────────────────────────────────────────────────────
print("  Generating VIZ 7: Pipeline architecture diagram...")

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')
fig.patch.set_facecolor('#0d1117')

stages = [
    (1.0,  '#1f6feb', '1. Raw Data\n(test.csv)',        'PRN filtering\n50% inactive rows'),
    (3.5,  '#388bfd', '2. Feature\nEngineering',         '47,744 timestamps\n30+ features'),
    (6.0,  '#8957e5', '3. Isolation\nForest',            'Unsupervised\npseudo-labels'),
    (8.5,  '#bc8cff', '4. 80/20 Split',                 'Train / Validate\nstratified'),
    (11.0, '#e3b341', '5. XGBoost\nClassifier',          'Threshold-tuned\nweighted F1'),
    (13.5, '#3fb950', '6. submission\n.csv',             '47,744 predictions\n+ confidence'),
]

for x, color, title, subtitle in stages:
    rect = plt.Rectangle((x - 0.85, 1.5), 1.7, 2.0,
                          facecolor=color + '22', edgecolor=color,
                          linewidth=1.5, zorder=2, transform=ax.transData)
    ax.add_patch(rect)
    ax.text(x, 2.85, title,    ha='center', va='center', fontsize=10,
            fontweight='bold', color=color, zorder=3)
    ax.text(x, 2.05, subtitle, ha='center', va='center', fontsize=8,
            color='#8b949e', zorder=3)

for i in range(len(stages) - 1):
    x1 = stages[i][0]     + 0.85
    x2 = stages[i + 1][0] - 0.85
    ax.annotate('', xy=(x2, 2.5), xytext=(x1, 2.5),
                arrowprops=dict(arrowstyle='->', color='#30363d',
                                lw=1.5, mutation_scale=15))

ax.set_title("Self-Training Pipeline: GNSS Spoofing Detection",
             fontsize=13, color='white', pad=8)
plt.tight_layout()
plt.savefig("plots/viz7_pipeline_architecture.png", dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.close()
print("  ✅ saved: plots/viz7_pipeline_architecture.png")


# ── Final summary ──────────────────────────────────────────
print("\n" + "="*60)
print("✅ ALL 7 PLOTS SAVED TO  plots/  FOLDER")
print("="*60)

  Generating VIZ 7: Pipeline architecture diagram...
  ✅ saved: plots/viz7_pipeline_architecture.png

✅ ALL 7 PLOTS SAVED TO  plots/  FOLDER
